# 04 — Baseline radiómico (PyRadiomics + clasificador shallow)

Implementa el Objetivo O2: extracción de features radiómicos con PyRadiomics, selección con LASSO, y entrenamiento de LogReg / RF / XGBoost. Evaluación con 5-fold CV estratificada **por paciente**.

Diseñado para correr en CPU; primer baseline cuantitativo del proyecto.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.data.inventory import build_inventory
from src.data.splits import make_stratified_group_kfold
from src.models.radiomics import RadiomicsPipeline, extract_features_for_record
from src.evaluation.metrics import binary_metrics_at_youden
from src.evaluation.bootstrap import bootstrap_ci, format_with_ci
from sklearn.metrics import roc_auc_score

DATA_ROOT = PROJECT_ROOT / 'data' / 'CirrMRI600plus_raw'
CACHE_DIR = PROJECT_ROOT / 'data' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

inventory = build_inventory(DATA_ROOT)
records = inventory.filter(task='binary')
print('records binarios:', len(records))

records binarios: 738


In [2]:
FEATS_CSV = CACHE_DIR / 'radiomics_features_binary.csv'
if FEATS_CSV.exists():
    feats_df = pd.read_csv(FEATS_CSV)
    print('features cargados de cache:', feats_df.shape)
else:
    rows = []
    for r in tqdm(records, desc='radiomics'):
        if r.mask_path is None:
            continue
        try:
            rows.append(extract_features_for_record(r))
        except Exception as exc:
            print(f'skip {r.patient_id}: {exc}')
    feats_df = pd.DataFrame(rows)
    feats_df.to_csv(FEATS_CSV, index=False)
    print('features extraídos:', feats_df.shape)
feats_df.head(3)

features cargados de cache: (738, 42)


,patient_id,modality,label,severity,fo_mean,fo_std,fo_median,fo_p10,fo_p90,fo_skew,...,lbp_5,lbp_6,lbp_7,lbp_8,lbp_9,shape_liver_volume_ml,shape_n_slices_with_liver,shape_bbox_volume_ml,shape_compactness,shape_axial_bbox_aspect_ratio
0,10,T1w,1,3.0,109.674786,30.323777,108.0,70.0,150.0,0.353857,...,0.153625,0.100539,0.080048,0.089874,0.124566,2138.379325,56.0,6363.196638,0.336054,0.938650
1,100,T1w,1,2.0,135.235117,20.537357,135.0,109.0,162.0,0.024521,...,0.169746,0.095732,0.069692,0.074662,0.101135,823.514719,38.0,2932.948784,0.280780,0.864286
2,101,T1w,1,1.0,353.911753,39.251607,352.0,313.0,403.0,-0.140290,...,0.174183,0.087749,0.052361,0.049543,0.064825,1393.234974,84.0,5781.267877,0.240991,1.048485


In [3]:
ids = feats_df['patient_id'].astype(str).tolist()
records_aligned = [r for r in records if r.patient_id in set(ids)]
# Mantener un único registro por (paciente, modalidad).
y_all = feats_df['label'].to_numpy().astype(int)
groups_all = feats_df['patient_id'].astype(str).to_numpy()
print('clase 1:', int((y_all == 1).sum()), '| clase 0:', int((y_all == 0).sum()))

clase 1: 628 | clase 0: 110


In [4]:
# Silenciar las ConvergenceWarning de LASSO/Lasso CV — son esperadas con features altamente correlacionadas y NO afectan el AUC final.
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', message='.*coordinate_descent.*')
warnings.filterwarnings('ignore', message='.*objective did not converge.*')

# Hiperparámetros por modelo (idénticos a configs/radiomics.yaml — el dataclass default solo sirve para LogReg).
MODEL_KWARGS = {
    'logreg': {'C': 1.0, 'class_weight': 'balanced', 'max_iter': 2000, 'solver': 'lbfgs'},
    'rf':     {'n_estimators': 500, 'max_depth': 8, 'class_weight': 'balanced', 'n_jobs': -1},
    'xgb':    {'n_estimators': 600, 'max_depth': 5, 'learning_rate': 0.05, 'scale_pos_weight': 0.18},
}

from sklearn.model_selection import StratifiedGroupKFold
results_by_model = {}
for model_name in ['logreg', 'rf', 'xgb']:
    fold_aucs = []
    oof = np.zeros(len(feats_df), dtype=float)
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (tr, va) in enumerate(sgkf.split(feats_df, y_all, groups=groups_all)):
        pipe = RadiomicsPipeline(
            k_features=30,
            selection='lasso',
            classifier_name=model_name,
            classifier_kwargs=MODEL_KWARGS[model_name],
        )
        pipe.fit(feats_df.iloc[tr], y_all[tr])
        proba = pipe.predict_proba(feats_df.iloc[va])[:, 1]
        oof[va] = proba
        auc = roc_auc_score(y_all[va], proba)
        fold_aucs.append(auc)
        print(f'  {model_name} fold {fold}: AUC={auc:.4f}')
    results_by_model[model_name] = {'fold_aucs': fold_aucs, 'oof': oof}
    print(f'{model_name} mean AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}')


  logreg fold 0: AUC=0.8481
  logreg fold 1: AUC=0.6717
  logreg fold 2: AUC=0.8766
  logreg fold 3: AUC=0.6691
  logreg fold 4: AUC=0.8156
logreg mean AUC: 0.7762 ± 0.0885
  rf fold 0: AUC=0.9044
  rf fold 1: AUC=0.6207
  rf fold 2: AUC=0.8747
  rf fold 3: AUC=0.6876
  rf fold 4: AUC=0.8351
rf mean AUC: 0.7845 ± 0.1107
  xgb fold 0: AUC=0.9019
  xgb fold 1: AUC=0.5948
  xgb fold 2: AUC=0.8906
  xgb fold 3: AUC=0.6673
  xgb fold 4: AUC=0.8753
xgb mean AUC: 0.7860 ± 0.1288


In [5]:
summary_rows = []
for name, res in results_by_model.items():
    bm = binary_metrics_at_youden(y_all, res['oof'])
    boot = bootstrap_ci(y_all, res['oof'], lambda yt, ys: roc_auc_score(yt, ys), n_resamples=1000, ci=0.95, seed=42)
    summary_rows.append({
        'model': name,
        'AUC': format_with_ci(boot.point, boot.ci_low, boot.ci_high),
        'AUC-PR': f'{bm.auc_pr:.3f}',
        'Sens@Y': f'{bm.sensitivity:.3f}',
        'Spec@Y': f'{bm.specificity:.3f}',
        'F1': f'{bm.f1:.3f}',
        'MCC': f'{bm.mcc:.3f}',
    })
summary = pd.DataFrame(summary_rows)
summary

,model,AUC,AUC-PR,Sens@Y,Spec@Y,F1,MCC
0,logreg,0.788 (95% CI 0.744–0.833),0.952,0.734,0.700,0.822,0.329
1,rf,0.774 (95% CI 0.730–0.816),0.947,0.596,0.836,0.733,0.308
2,xgb,0.781 (95% CI 0.739–0.822),0.953,0.637,0.773,0.760,0.295


### Resultado esperado
AUC en el rango 0.70–0.82 con CI 95% no incluyendo 0.5. Este es el baseline contra el cual debe ganar el modelo DL en el notebook 05 (≥ +5 puntos de AUC con DeLong p<0.05).

In [6]:
out_dir = PROJECT_ROOT / 'reports' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / 'radiomics_results.csv', index=False)
for name, res in results_by_model.items():
    pred_df = pd.DataFrame({'patient_id': feats_df['patient_id'], 'modality': feats_df['modality'], 'label': y_all, 'proba': res['oof']})
    pred_df.to_csv(out_dir / f'radiomics_{name}_preds.csv', index=False)
print('tablas guardadas en', out_dir)

tablas guardadas en c:\Users\Sebas\Desktop\Talleres IyV\cirrosis-detection\reports\tables
